# Faisabilité de la modélisation

## La question

Peut-on, avec les données déjà en base, **appliquer des modèles statistiques ou d'IA aux
courses** ? Autrement dit : y a-t-il assez d'exemples, assez de variables connues *avant le
départ*, et assez de lien entre ces variables et le résultat, pour que ce soit autre chose
qu'un pile ou face habillé ?

Ce document ne construit pas un modèle de production. Il teste les conditions préalables :
définition d'une cible, disponibilité des informations, corrélations, et un modèle linéaire
très simple pour voir si l'on dépasse les évidences (la cote, le favori).

La période est la même que celle du panorama : du **1er janvier 2025 au 31 août 2026**.
Les relevés de cotes en direct ne font pas partie du périmètre.

## Sommaire

1. [Ce que l'on peut chercher à prédire](#cibles)
2. [Ce qui est connu avant le départ](#avant-depart)
3. [Le jeu de données](#jeu)
4. [Quelles variables sont liées au résultat ?](#correlations)
5. [La cote absorbe-t-elle déjà tout ?](#cote)
6. [Un modèle simple, pour voir](#modele)
7. [Ce qui rend l'exercice délicat](#limites)
8. [Verdict](#verdict)

<a id="cibles"></a>
## 1. Ce que l'on peut chercher à prédire

Trois cibles raisonnables se dessinent, de la plus simple à la plus ambitieuse.

| Cible | Question posée au modèle | Intérêt |
| --- | --- | --- |
| **Gagnant** | ce cheval finit-il premier ? | claire, mais rare : environ un cheval sur douze |
| **Placé** | finit-il dans les trois premiers ? | plus fréquente, utile pour les paris « placé » |
| **Rang** | dans quel ordre arrivent-ils ? | la plus riche, la plus difficile |

Deux lectures du succès ne doivent pas être confondues.

- **Classer les chevaux** : retrouver le vainqueur plus souvent que le hasard, ou mieux que
  « je prends le favori ». C'est un problème de statistique.
- **Battre le marché** : miser et gagner de l'argent. Ici la cote n'est plus une simple
  variable : c'est le prix. Un modèle peut très bien classer sans être rentable, parce que
  le PMU prélève déjà sa part sur chaque enjeu.

La suite du document sépare ces deux exigences.

<a id="avant-depart"></a>
## 2. Ce qui est connu avant le départ

In [ ]:
import sys
import warnings
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "pmu").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import design_system as ds
from pmu.types.types import ProgrammeIdentifier
from pmu.utils import programme_date_to_date

warnings.filterwarnings("ignore", category=FutureWarning)
THEME = ds.apply_theme("light")

start_date = ProgrammeIdentifier("01012025")
end_date = ProgrammeIdentifier("31082026")
debut = pd.Timestamp(programme_date_to_date(str(start_date)))
fin = pd.Timestamp(programme_date_to_date(str(end_date)))

In [ ]:
ESPACE = "\u202f"


def nombre(valeur, decimales=0, unite=""):
    if pd.isna(valeur):
        return "—"
    texte = f"{valeur:,.{decimales}f}".replace(",", ESPACE).replace(".", ",")
    return f"{texte}{unite}"


def tableau(frame, titre=None, formats=None, barres=None, gradient=None, precision=0):
    styler = frame.style.set_table_styles(ds.table_css("light"))
    if titre:
        styler = styler.set_caption(titre)
    if formats:
        styler = styler.format(formats, na_rep="—")
    else:
        styler = styler.format(precision=precision, thousands=ESPACE, decimal=",", na_rep="—")
    if barres:
        styler = styler.bar(subset=barres, color=THEME["chart_3"], align="left", height=62, width=96)
    if gradient:
        styler = styler.background_gradient(cmap=ds.diverging_cmap("light"), subset=gradient, vmin=-1, vmax=1)
    return styler


def chiffres_cles(valeurs: dict, titre: str):
    frame = pd.DataFrame({"": list(valeurs.values())}, index=list(valeurs.keys()))
    return tableau(frame, titre, formats={"": "{}"})


def figure(titre: str, figsize=(11, 4.4)):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_title(titre)
    return fig, ax


def barh_ax(ax, serie, titre=None, fmt="{:,.2f}", couleur=None):
    ordre = serie.sort_values()
    ax.barh(
        [str(index) for index in ordre.index],
        ordre.to_numpy(dtype=float),
        height=0.68,
        color=couleur or THEME["primary"],
    )
    ds.annotate_bars(ax, orientation="horizontal", fmt=fmt)
    ds.rounded_bars(ax, orientation="horizontal")
    ax.grid(axis="x")
    ax.grid(axis="y", visible=False)
    etendue = float(max(abs(ordre.min()), abs(ordre.max())))
    ax.set_xlim(-etendue * 1.2, etendue * 1.2) if (ordre.min() < 0 < ordre.max()) else ax.set_xlim(
        0, float(ordre.max()) * 1.18
    )
    if titre:
        ax.set_title(titre)
    return ax


def barres(serie, titre, xlabel="", fmt="{:,.2f}", couleur=None, hauteur=0.36):
    fig, ax = figure(titre, (10.5, max(2.8, hauteur * len(serie) + 1.5)))
    if serie.min() < 0 < serie.max():
        ordre = serie.sort_values()
        couleurs = [THEME["chart_2"] if valeur < 0 else THEME["primary"] for valeur in ordre]
        ax.barh([str(i) for i in ordre.index], ordre.to_numpy(dtype=float), height=0.68, color=couleurs)
        ds.annotate_bars(ax, orientation="horizontal", fmt=fmt)
        ds.rounded_bars(ax, orientation="horizontal")
        ax.axvline(0, color=THEME["foreground"], linewidth=0.8)
        etendue = float(max(abs(ordre.min()), abs(ordre.max())))
        ax.set_xlim(-etendue * 1.25, etendue * 1.25)
        ax.grid(axis="x")
        ax.grid(axis="y", visible=False)
    else:
        barh_ax(ax, serie, fmt=fmt, couleur=couleur)
    ax.set_xlabel(xlabel)
    return fig, ax

Avant de chercher des corrélations, il faut tracer une ligne nette : **ce que le modèle a le
droit de voir** au moment où l'on voudrait l'interroger, et ce qui n'existe qu'une fois
l'arrivée connue. Utiliser le chrono ou la place pour « prédire » le gagnant n'aurait aucun
sens : ce serait tricher.

In [ ]:
disponibilite = pd.DataFrame(
    [
        ("Âge, sexe, race", "Oui", "Toujours", "Profil du cheval"),
        ("Jockey / driver et entraîneur", "Oui", "Toujours", "Acteurs"),
        ("Palmarès (courses, victoires, places)", "Oui", "Toujours", "Forme passée"),
        ("Gains en carrière", "Oui", "Presque toujours", "Forme passée"),
        ("Musique (historique de résultats)", "Oui", "Presque toujours", "Forme récente"),
        ("Cote de la veille", "Oui", "Souvent", "Marché, la veille"),
        ("Cote au départ", "Oui, mais tardive", "Très souvent", "Marché, juste avant le top"),
        ("Place à la corde", "Oui", "Galop et trot attelé", "Départ"),
        ("Poids, distance de handicap", "Oui", "Selon la discipline", "Conditions de course"),
        ("Déferrage, œillères", "Oui", "Surtout le trot", "Équipement"),
        ("Distance, discipline, hippodrome", "Oui", "Toujours", "Cadre de l'épreuve"),
        ("Météo de la réunion", "Oui", "Presque toujours", "Contexte"),
        ("Souplesse du terrain", "Oui", "Une course sur trois", "Contexte"),
        ("Place à l'arrivée", "Non — c'est la cible", "Après la course", "Résultat"),
        ("Incident, chrono, écart", "Non", "Après la course", "Résultat"),
    ],
    columns=["Information", "Disponible avant le départ", "Couverture", "Famille"],
).set_index("Information")
display(
    tableau(
        disponibilite,
        "Ce qu'un modèle a le droit de voir, et ce qu'il ne doit pas voir",
        formats={colonne: "{}" for colonne in disponibilite.columns},
    )
)

**À retenir.** Le socle « avant départ » est large : identité, palmarès, acteurs, cadre de
l'épreuve, et une cote. La cote au départ est légitime si l'on prédit *à la dernière minute* ;
elle l'est moins si l'on veut un avis plus tôt dans la journée — dans ce cas, seule la cote de
la veille reste honnête. Tout ce qui décrit l'arrivée (place, incident, chrono) est réservé à
la cible, jamais aux variables.

<a id="jeu"></a>
## 3. Le jeu de données

On ne garde que les **partants effectifs** des courses effectivement arrivées. Un cheval
annoncé puis retiré n'apprend rien au modèle sur le résultat sportif. Une course encore
programmée n'a pas de cible.

In [ ]:
from dataset.load_jeu_modelisation import load_jeu_modelisation

brut = load_jeu_modelisation(start_date, end_date)
brut["gains_euros"] = brut["gains_carriere"] / 100
brut["taux_victoire"] = np.where(
    brut["nombre_courses"] > 0, brut["nombre_victoires"] / brut["nombre_courses"], np.nan
)
brut["taux_place"] = np.where(
    brut["nombre_courses"] > 0, brut["nombre_places"] / brut["nombre_courses"], np.nan
)
brut["log_gains"] = np.log10(brut["gains_euros"].clip(lower=1))
brut["cote_utile"] = brut["cote_directe"].where(brut["cote_directe"].between(1, 998))
brut["cote_veille"] = brut["cote_reference"].where(brut["cote_reference"].between(1, 998))
brut["log_cote"] = np.log(brut["cote_utile"])
brut["log_cote_veille"] = np.log(brut["cote_veille"])
brut["probabilite_cote"] = 1 / brut["cote_utile"]
brut["gagnant"] = brut["ordre_arrivee"].eq(1)
brut["place"] = brut["ordre_arrivee"].between(1, 3)

courses_avec_vainqueur = brut.loc[brut["gagnant"], "course_id"].unique()
jeu = brut[brut["course_id"].isin(courses_avec_vainqueur)].copy()

display(
    chiffres_cles(
        {
            "Période": f"{debut:%d/%m/%Y} → {fin:%d/%m/%Y}",
            "Partants des courses arrivées": nombre(len(jeu)),
            "Courses avec un vainqueur": nombre(jeu["course_id"].nunique()),
            "Chevaux différents": nombre(jeu["nom"].nunique()),
            "Part de gagnants": f"{nombre(jeu['gagnant'].mean() * 100, 2)} %",
            "Part de placés": f"{nombre(jeu['place'].mean() * 100, 2)} %",
            "Cote au départ renseignée": f"{nombre(jeu['cote_utile'].notna().mean() * 100, 1)} %",
            "Cote de la veille renseignée": f"{nombre(jeu['cote_veille'].notna().mean() * 100, 1)} %",
        },
        "Le terrain de jeu de la modélisation",
    )
)

In [ ]:
par_discipline = (
    jeu.groupby("discipline", observed=True)
    .agg(
        courses=("course_id", "nunique"),
        partants=("id", "size"),
        partants_par_course=("nombre_partants", "mean"),
        taux_victoire_moyen=("gagnant", "mean"),
        cote_mediane=("cote_utile", "median"),
        corde_renseignee=("place_corde", lambda s: s.notna().mean() * 100),
    )
    .sort_values("courses", ascending=False)
)
par_discipline.columns = [
    "Courses",
    "Partants",
    "Chevaux par course",
    "Un gagnant parmi (%)",
    "Cote médiane",
    "Place à la corde renseignée (%)",
]
par_discipline["Un gagnant parmi (%)"] *= 100
display(
    tableau(
        par_discipline,
        "Volume et densité par discipline : autant de sous-problèmes que de disciplines",
        formats={
            "Courses": lambda v: nombre(v),
            "Partants": lambda v: nombre(v),
            "Chevaux par course": lambda v: nombre(v, 1),
            "Un gagnant parmi (%)": lambda v: nombre(v, 1),
            "Cote médiane": lambda v: nombre(v, 1),
            "Place à la corde renseignée (%)": lambda v: nombre(v, 1),
        },
        barres=["Courses"],
    )
)

**À retenir.** On dispose de plusieurs dizaines de milliers de courses et de plusieurs centaines
de milliers de partants : **le volume n'est pas le frein**. En revanche, le plat et le trot
attelé pèsent l'essentiel des exemples ; le cross ou le trot monté sont trop peu nombreux pour
un modèle à part, sauf à les regrouper. La place à la corde, utile en plat, est quasi absente
ailleurs : un modèle unique « toutes disciplines » mélange des informations qui n'ont pas le
même sens.

<a id="correlations"></a>
## 4. Quelles variables sont liées au résultat ?

On mesure ici le lien entre chaque variable numérique et le fait de gagner, d'être placé, ou
le rang d'arrivée. Le coefficient de Spearman décrit un lien monotone : « quand cette grandeur
monte, le cheval gagne-t-il plus souvent ? ». Il ne dit pas *pourquoi*, et il ne dit pas si
l'information est déjà dans la cote.

In [ ]:
VARIABLES = {
    "log_cote": "Cote au départ (log)",
    "log_cote_veille": "Cote de la veille (log)",
    "probabilite_cote": "Chances affichées (1 / cote)",
    "age": "Âge",
    "nombre_courses": "Courses déjà disputées",
    "taux_victoire": "Taux de victoire en carrière",
    "taux_place": "Taux de places en carrière",
    "log_gains": "Gains en carrière (log)",
    "place_corde": "Place à la corde",
    "handicap_poids": "Poids de handicap",
    "handicap_distance": "Distance de handicap",
    "nombre_partants": "Taille du peloton",
    "distance": "Distance de l'épreuve",
    "temperature": "Température",
    "penetrometre": "Souplesse du terrain",
}

cibles = {"gagnant": "Gagnant", "place": "Placé (top 3)", "ordre_arrivee": "Rang d'arrivée"}
liens = pd.DataFrame(index=list(VARIABLES.values()))
for cible, libelle_cible in cibles.items():
    liens[libelle_cible] = [
        jeu[[colonne, cible]].dropna().corr(method="spearman").iloc[0, 1]
        for colonne in VARIABLES
    ]
liens["Couverture (%)"] = [jeu[colonne].notna().mean() * 100 for colonne in VARIABLES]
display(
    tableau(
        liens,
        "Lien (Spearman) entre chaque variable et le résultat",
        formats={
            "Gagnant": lambda v: nombre(v, 3),
            "Placé (top 3)": lambda v: nombre(v, 3),
            "Rang d'arrivée": lambda v: nombre(v, 3),
            "Couverture (%)": lambda v: nombre(v, 1),
        },
        gradient=["Gagnant", "Placé (top 3)", "Rang d'arrivée"],
        barres=["Couverture (%)"],
    )
)

In [ ]:
fig, ax = barres(
    liens["Gagnant"],
    "Force du lien avec la victoire",
    "Coefficient de Spearman (négatif = plus la valeur est haute, moins le cheval gagne)",
    fmt="{:,.2f}",
)
plt.show()

In [ ]:
entre_elles = jeu[list(VARIABLES)].rename(columns=VARIABLES).corr(method="spearman")
fig, ax = plt.subplots(figsize=(10.5, 8.2))
image = ax.imshow(entre_elles.to_numpy(), cmap=ds.diverging_cmap("light"), vmin=-1, vmax=1)
ax.set_xticks(range(len(entre_elles.columns)), entre_elles.columns, rotation=55, ha="right")
ax.set_yticks(range(len(entre_elles.index)), entre_elles.index)
ax.set_title("Les variables se répètent-elles entre elles ?")
ax.grid(visible=False)
barre = fig.colorbar(image, ax=ax, pad=0.02)
barre.outline.set_visible(False)
plt.show()

Les variables qualitatives se lisent autrement : on compare le taux de victoire d'un groupe à
celui de l'ensemble.

In [ ]:
def ecart_groupes(colonne, minimum=3000):
    groupes = jeu.groupby(colonne, observed=True)["gagnant"].agg(["mean", "size"])
    groupes = groupes[groupes["size"] >= minimum]
    global_rate = jeu["gagnant"].mean()
    groupes["ecart"] = (groupes["mean"] - global_rate) * 100
    groupes["taux"] = groupes["mean"] * 100
    return groupes.sort_values("ecart", ascending=False)


blocs = []
for colonne, titre in [
    ("sexe", "Sexe"),
    ("discipline", "Discipline"),
    ("inedit", "Inédit"),
    ("driver_change", "Changement de driver"),
    ("favori", "Marqué favori"),
]:
    groupe = ecart_groupes(colonne, minimum=800)
    groupe = groupe.rename_axis("Modalité").reset_index()
    groupe.insert(0, "Variable", titre)
    groupe["Modalité"] = [
        {True: "Oui", False: "Non"}.get(valeur, str(valeur)) for valeur in groupe["Modalité"]
    ]
    blocs.append(groupe[["Variable", "Modalité", "size", "taux", "ecart"]])
qualitatif = pd.concat(blocs, ignore_index=True).set_index(["Variable", "Modalité"])
qualitatif.columns = ["Partants", "Victoires (%)", "Écart au hasard (pts)"]
display(
    tableau(
        qualitatif,
        "Taux de victoire selon quelques catégories",
        formats={
            "Partants": lambda v: nombre(v),
            "Victoires (%)": lambda v: nombre(v, 2),
            "Écart au hasard (pts)": lambda v: nombre(v, 2),
        },
    )
)

**À retenir.** Un signal existe, et il n'est pas fantôme.

La **cote** (et son inverse, les chances affichées) est de loin la variable la plus liée au
résultat. Viennent ensuite le palmarès — taux de victoire, taux de places, gains — puis, plus
faiblement, la place à la corde et l'âge. La météo et le pénétromètre, eux, ne disent presque
rien du *vainqueur* : ils décrivent plutôt comment la course se court, pas qui la gagne.

Plusieurs variables de forme se recoupent (gains, victoires, nombre de courses). Les garder
toutes dans un modèle reviendrait à raconter trois fois la même histoire. Le marqueur « favori »
est presque redondant avec la cote : c'est une étiquette posée sur les chevaux déjà les moins
cotés.

<a id="cote"></a>
## 5. La cote absorbe-t-elle déjà tout ?

C'est la question décisive. Si, une fois la cote connue, plus rien d'autre n'aide à trouver le
gagnant, alors un modèle d'IA n'a d'intérêt que s'il vise autre chose que « battre le marché à
la dernière minute » : expliquer, classer plus tôt, ou travailler une discipline mal cotée.

Pour le voir, on retire de chaque cheval ce que la cote avait déjà prévu, et l'on cherche s'il
reste un lien avec le palmarès, la corde, l'âge.

In [ ]:
residuel = jeu.dropna(subset=["cote_utile"]).copy()
residuel["surprise"] = residuel["gagnant"].astype(float) - residuel["probabilite_cote"]

variables_hors_cote = {
    "taux_victoire": "Taux de victoire",
    "taux_place": "Taux de places",
    "log_gains": "Gains (log)",
    "nombre_courses": "Expérience",
    "age": "Âge",
    "place_corde": "Place à la corde",
    "nombre_partants": "Taille du peloton",
    "distance": "Distance",
    "temperature": "Température",
}

residus = pd.Series(
    {
        libelle: residuel[[colonne, "surprise"]].dropna().corr(method="spearman").iloc[0, 1]
        for colonne, libelle in variables_hors_cote.items()
    },
    name="Lien avec la « surprise »",
)
display(
    tableau(
        residus.to_frame(),
        "Une fois la cote retirée, que reste-t-il ?",
        formats={"Lien avec la « surprise »": lambda v: nombre(v, 3)},
    )
)

In [ ]:
fig, ax = barres(
    residus,
    "Signal restant après avoir retiré la prévision de la cote",
    "Spearman avec (victoire − 1 / cote)",
    fmt="{:,.3f}",
)
plt.show()

In [ ]:
# La cote de la veille suffit-elle, ou faut-il attendre le départ ?
deux_cotes = jeu.dropna(subset=["cote_utile", "cote_veille"]).copy()
accord = deux_cotes[["cote_veille", "cote_utile"]].corr(method="spearman").iloc[0, 1]
display(
    chiffres_cles(
        {
            "Chevaux avec les deux cotes": nombre(len(deux_cotes)),
            "Accord veille / départ": nombre(accord, 3),
            "Lien veille → victoire": nombre(
                deux_cotes[["log_cote_veille", "gagnant"]].corr(method="spearman").iloc[0, 1], 3
            ),
            "Lien départ → victoire": nombre(
                deux_cotes[["log_cote", "gagnant"]].corr(method="spearman").iloc[0, 1], 3
            ),
        },
        "Prévoir la veille, ou attendre le départ ?",
    )
)

**À retenir.** La cote avale l'essentiel du signal utile pour *trouver le gagnant*. Ce qui
reste ensuite n'est pas du bruit, mais ce n'est plus le même message : les chevaux au fort
palmarès gagnent **un peu moins souvent que leur cote ne le promet**. Le marché les surévalue.
La place à la corde, elle, garde un petit effet que le prix n'intègre pas tout à fait.

Autrement dit, le reliquat ne sert pas à mieux désigner le favori : il sert, éventuellement, à
trouver des écarts de prix. C'est une piste pour un modèle de pari, pas pour un modèle de
classement.

La cote de la veille est déjà très proche de celle du départ. Un modèle interrogé le matin a
donc presque autant d'information marché qu'un modèle interrogé à la dernière minute, avec un
léger manque à gagner.

<a id="modele"></a>
## 6. Un modèle simple, pour voir

Un modèle linéaire, volontairement naïf, permet de passer des corrélations à une question plus
concrète : **sur des courses que le modèle n'a jamais vues**, retrouve-t-il le vainqueur mieux
que le hasard, mieux que le favori, mieux que la cote seule ?

La coupure est temporelle : on apprend sur 2025, on teste sur 2026. C'est la seule coupure
honnête — mélanger les dates reviendrait à se servir du futur pour prédire le passé.

In [ ]:
def preparer(frame):
    table = frame.dropna(subset=["cote_utile", "taux_victoire", "age", "nombre_partants"]).copy()
    table["place_corde_connue"] = table["place_corde"].notna().astype(float)
    table["place_corde"] = table["place_corde"].fillna(table["nombre_partants"] / 2)
    table["log_partants"] = np.log(table["nombre_partants"])
    return table


def predire(entrainement, test, colonnes):
    pipeline = Pipeline(
        [
            ("echelle", StandardScaler()),
            (
                "modele",
                LogisticRegression(max_iter=400, class_weight="balanced"),
            ),
        ]
    )
    pipeline.fit(entrainement[colonnes], entrainement["gagnant"].astype(int))
    return pipeline.predict_proba(test[colonnes])[:, 1], pipeline


def precision_course(frame, score, nom_score):
    ordre = frame.assign(_score=score).sort_values("_score", ascending=False)
    choix = ordre.groupby("course_id", observed=True).first()
    return choix["gagnant"].mean()


def retour_pari(frame, score):
    ordre = frame.assign(_score=score).sort_values("_score", ascending=False)
    choix = ordre.groupby("course_id", observed=True).first()
    return np.where(choix["gagnant"], choix["cote_utile"], 0.0).mean()


complet = preparer(jeu)
train = complet[complet["date_programme"] < "2026-01-01"]
test = complet[complet["date_programme"] >= "2026-01-01"]

socle = ["log_cote"]
profil = ["age", "taux_victoire", "taux_place", "log_gains", "nombre_courses", "place_corde", "place_corde_connue", "log_partants"]
complet_vars = socle + profil

proba_cote, _ = predire(train, test, socle)
proba_profil, _ = predire(train, test, profil)
proba_plein, modele_plein = predire(train, test, complet_vars)

favori_test = test.sort_values("cote_utile").groupby("course_id", observed=True).first()

bilan_modele = pd.DataFrame(
    {
        "AUC (cheval)": [
            0.5,
            roc_auc_score(test["gagnant"], 1 / test["cote_utile"]),
            roc_auc_score(test["gagnant"], proba_cote),
            roc_auc_score(test["gagnant"], proba_profil),
            roc_auc_score(test["gagnant"], proba_plein),
        ],
        "Vainqueur trouvé": [
            (1 / test.groupby("course_id")["id"].transform("size")).groupby(test["course_id"]).first().mean(),
            favori_test["gagnant"].mean(),
            precision_course(test, proba_cote, "cote"),
            precision_course(test, proba_profil, "profil"),
            precision_course(test, proba_plein, "plein"),
        ],
        "Retour pour 1 € misé": [
            np.nan,
            np.where(favori_test["gagnant"], favori_test["cote_utile"], 0.0).mean(),
            retour_pari(test, proba_cote),
            retour_pari(test, proba_profil),
            retour_pari(test, proba_plein),
        ],
    },
    index=[
        "Hasard (un cheval au hasard)",
        "Règle du favori (plus basse cote)",
        "Modèle : cote seule",
        "Modèle : profil, sans la cote",
        "Modèle : cote + profil",
    ],
)
display(
    chiffres_cles(
        {
            "Courses d'apprentissage (2025)": nombre(train["course_id"].nunique()),
            "Partants d'apprentissage": nombre(len(train)),
            "Courses de test (2026)": nombre(test["course_id"].nunique()),
            "Partants de test": nombre(len(test)),
        },
        "Coupure temporelle",
    )
)
display(
    tableau(
        bilan_modele,
        "Ce que retrouve un modèle linéaire, sur des courses qu'il n'a pas vues",
        formats={
            "AUC (cheval)": lambda v: nombre(v, 3),
            "Vainqueur trouvé": lambda v: nombre(v * 100, 1, " %"),
            "Retour pour 1 € misé": lambda v: nombre(v, 3, " €") if pd.notna(v) else "—",
        },
    )
)

In [ ]:
poids = pd.Series(
    modele_plein.named_steps["modele"].coef_[0],
    index=[
        "Cote (log)",
        "Âge",
        "Taux de victoire",
        "Taux de places",
        "Gains (log)",
        "Expérience",
        "Place à la corde",
        "Corde renseignée",
        "Taille du peloton (log)",
    ],
)
fig, ax = barres(
    poids,
    "Ce à quoi le modèle s'accroche (poids, après mise à l'échelle)",
    "Poids positif = favorise la victoire",
    fmt="{:,.2f}",
)
plt.show()

In [ ]:
par_disc = []
for discipline, morceau in test.groupby("discipline", observed=True):
    if morceau["course_id"].nunique() < 200:
        continue
    proba, _ = predire(train[train["discipline"] == discipline], morceau, complet_vars)
    if len(train[train["discipline"] == discipline]) < 500:
        continue
    favoris = morceau.sort_values("cote_utile").groupby("course_id", observed=True).first()
    par_disc.append(
        {
            "Discipline": discipline,
            "Courses testées": morceau["course_id"].nunique(),
            "Favori trouve le vainqueur": favoris["gagnant"].mean() * 100,
            "Modèle trouve le vainqueur": precision_course(morceau, proba, discipline) * 100,
        }
    )
comparaison = pd.DataFrame(par_disc).set_index("Discipline").sort_values("Courses testées", ascending=False)
display(
    tableau(
        comparaison,
        "Le modèle ajoute-t-il quelque chose, discipline par discipline ?",
        formats={
            "Courses testées": lambda v: nombre(v),
            "Favori trouve le vainqueur": lambda v: nombre(v, 1, " %"),
            "Modèle trouve le vainqueur": lambda v: nombre(v, 1, " %"),
        },
    )
)

**À retenir.** Trois résultats se dessinent, plus importants que le modèle lui-même.

Premier : **sans la cote, le profil du cheval n'est pas muet**. Âge, palmarès et corde
suffisent déjà à classer mieux que le hasard. Il y a donc de l'information dans la base, y
compris pour un avis formulé sans regarder le marché.

Deuxième : **dès que la cote est dans le modèle, le reste n'ajoute presque rien** au
classement. Le modèle « cote + profil » confond avec la règle du favori. C'est cohérent
avec la section précédente : pour désigner un vainqueur, le marché a déjà fait le travail.

Troisième : **aucun de ces modèles n'est rentable**. Retrouver le vainqueur un peu plus souvent
ne suffit pas : les cotes des chevaux que l'on choisit alors sont moins généreuses, et le
prélèvement reste. Battre le marché demanderait un signal que le prix n'a pas encore, ou une
cible différente (trouver des *outsiders* sous-évalués, pas le favori un peu mieux).

<a id="limites"></a>
## 7. Ce qui rend l'exercice délicat

In [ ]:
limites = pd.DataFrame(
    [
        (
            "Les exemples ne sont pas indépendants",
            "Le même cheval, le même driver, le même entraîneur reviennent des dizaines de fois. "
            "Un modèle qui « reconnaît » un nom plutôt qu'une situation surestime ses performances.",
        ),
        (
            "Le gagnant est rare",
            "Environ 8 à 10 % des lignes sont des victoires. Un modèle naïf peut paraître bon "
            "en ne pariant jamais sur personne. Il faut juger au niveau de la course, pas seulement "
            "ligne à ligne.",
        ),
        (
            "Chaque discipline est un sport différent",
            "La corde compte en plat, le déferrage au trot, le poids à l'obstacle. Un seul modèle "
            "pour tout le programme noie ces effets.",
        ),
        (
            "Des informations manquent de façon structurée",
            "Le terrain, le déferrage, parfois la cote de la veille : l'absence n'est pas le fruit "
            "du hasard, elle signale une discipline ou un pays. Il ne faut pas les « remplir » "
            "sans précaution.",
        ),
        (
            "La cote au départ n'est pas un avis du matin",
            "S'en servir, c'est se placer juste avant le top. Pour un pronostic plus tôt, seule "
            "la cote de la veille — un peu moins précise — est honnête.",
        ),
        (
            "On n'a pas l'histoire de la cote dans la journée",
            "Le marché bouge. Sans ces mouvements, on ne peut pas travailler les stratégies de "
            "timing, seulement une photo à un instant.",
        ),
        (
            "Les montants étrangers mélangent les devises",
            "Une dotation « énorme » à Hong Kong n'est pas comparable à Longchamp. Cette variable "
            "trompe un modèle s'il mélange les pays.",
        ),
    ],
    columns=["Point de vigilance", "Pourquoi ça compte"],
).set_index("Point de vigilance")
display(
    tableau(
        limites,
        "Ce qu'il faudra traiter avant de parler d'IA en production",
        formats={"Pourquoi ça compte": "{}"},
    )
)

**À retenir.** Rien de tout cela n'interdit la modélisation. Cela interdit seulement de coller
un algorithme sur la table brute et d'en croire le premier score. Le travail utile est en amont :
séparer les disciplines, calculer les performances passées *sans regarder le futur*, juger les
modèles course par course, et décider clairement si l'on cherche à classer ou à miser.

<a id="verdict"></a>
## 8. Verdict

In [ ]:
verdict = pd.DataFrame(
    [
        ("Classer les chevaux d'une course", "Oui", "Volume suffisant, cible claire, signal mesurable."),
        ("Estimer une probabilité de victoire", "Oui", "La cote est déjà une bonne base ; le profil l'affine un peu."),
        ("Le faire discipline par discipline", "Oui, recommandé", "Le plat et le trot attelé ont assez d'exemples."),
        ("Le faire dès la veille", "Oui, avec un peu moins de précision", "La cote de la veille est très proche de celle du départ."),
        ("Remplacer la cote par de l'IA", "Non", "Sans la cote, on fait mieux que le hasard, nettement moins bien que le marché."),
        ("Battre le marché avec ce socle seul", "Pas démontré", "Le modèle simple n'est pas rentable ; il faudrait un signal hors prix."),
        ("Modéliser le mouvement des cotes", "Pas avec ces données", "On n'a qu'une photo, pas le film de la journée."),
        ("Un modèle unique pour toutes les courses", "Déconseillé", "Les disciplines ne parlent pas la même langue."),
    ],
    columns=["Usage envisagé", "Possible ?", "Commentaire"],
).set_index("Usage envisagé")
display(
    tableau(
        verdict,
        "Alors : peut-on appliquer des modèles à ces courses ?",
        formats={"Possible ?": "{}", "Commentaire": "{}"},
    )
)

**Oui, on peut appliquer des modèles statistiques ou d'IA à ces courses** — à condition de viser
le bon problème.

La base a le volume, la cible, et des variables connues avant le départ. Un lien réel relie le
profil du cheval au résultat. Ce n'est pas du bruit. En revanche, **la cote est déjà un modèle**,
et un modèle très bon : tout ce que le palmarès, l'âge ou la corde ont à dire, le marché l'a
presque entièrement intégré. L'IA a donc de la valeur pour *classer*, *expliquer*, *anticiper
dès la veille*, ou travailler une discipline à part. Elle n'en a presque pas pour simplement
répéter la cote à la dernière seconde, et elle n'a pas encore, dans ce socle, de quoi promettre
un pari rentable.

La suite naturelle n'est pas un algorithme plus compliqué sur la même table. C'est un jeu plus
propre : un modèle par discipline, des performances passées calculées dans le temps, une
évaluation course par course — et, si l'ambition est de miser, la chasse à un signal que le
prix n'a pas déjà avalé.